In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import sqlite3
import pandas as pd
import flwr as fl
import threading
from collections import OrderedDict

# 1. Setup Hardware Acceleration and Model Architecture
device = torch.device("cpu")

class DiagnosticNet(nn.Module):
    def __init__(self):
        super(DiagnosticNet, self).__init__()
        self.layer1 = nn.Linear(30, 16)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(16, 8)
        self.output = nn.Linear(8, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.sigmoid(self.output(x))
        return x

# 2. Define the Federated Hospital Client
class HospitalClient(fl.client.NumPyClient):
    def __init__(self, hospital_name):
        self.hospital_name = hospital_name
        self.model = DiagnosticNet().to(device)
        
        # Load local data from the isolated SQLite database
        conn = sqlite3.connect(f"privacare_dbs/{self.hospital_name}.db")
        train_df = pd.read_sql("SELECT * FROM train_data", conn)
        test_df = pd.read_sql("SELECT * FROM test_data", conn)
        conn.close()
        
        # Convert to PyTorch Tensors
        self.X_train = torch.tensor(train_df.drop(columns=["target"]).values, dtype=torch.float32).to(device)
        self.y_train = torch.tensor(train_df["target"].values, dtype=torch.float32).view(-1, 1).to(device)
        self.X_test = torch.tensor(test_df.drop(columns=["target"]).values, dtype=torch.float32).to(device)
        self.y_test = torch.tensor(test_df["target"].values, dtype=torch.float32).view(-1, 1).to(device)

    def get_parameters(self, config):
        return [val.cpu().numpy() for _, val in self.model.state_dict().items()]

    def set_parameters(self, parameters):
        params_dict = zip(self.model.state_dict().keys(), parameters)
        state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
        self.model.load_state_dict(state_dict, strict=True)

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        optimizer = optim.Adam(self.model.parameters(), lr=0.005)
        criterion = nn.BCELoss()
        
        # Local Training Loop
        self.model.train()
        for _ in range(5): # Train for 5 local epochs
            optimizer.zero_grad()
            outputs = self.model(self.X_train)
            loss = criterion(outputs, self.y_train)
            loss.backward()
            optimizer.step()
            
        # Apply Differential Privacy (Gaussian Noise) before returning weights
        with torch.no_grad():
            for param in self.model.parameters():
                noise = torch.normal(mean=0.0, std=0.05, size=param.size()).to(device)
                param.add_(noise)
                
        return self.get_parameters(config={}), len(self.X_train), {}

    def evaluate(self, parameters, config):
        self.set_parameters(parameters)
        criterion = nn.BCELoss()
        self.model.eval()
        with torch.no_grad():
            outputs = self.model(self.X_test)
            loss = criterion(outputs, self.y_test).item()
            preds = (outputs >= 0.5).float()
            accuracy = (preds == self.y_test).float().mean().item()
        return loss, len(self.X_test), {"accuracy": accuracy}

# 3. Launch Hospital Nodes Concurrently
def start_node(name):
    print(f"[{name}] Connecting to central server...")
    fl.client.start_client(server_address="127.0.0.1:8080", client=HospitalClient(name).to_client())

for h_name in ["hospital_A", "hospital_B", "hospital_C"]:
    threading.Thread(target=start_node, args=(h_name,)).start()

[hospital_A] Connecting to central server...[hospital_B] Connecting to central server...

[hospital_C] Connecting to central server...


	Instead, use the `flower-supernode` CLI command to start a SuperNode as shown below:

		$ flower-supernode --insecure --superlink='<IP>:<PORT>'

	To view all available options, run:

		$ flower-supernode --help

	Using `start_client()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
	Instead, use the `flower-supernode` CLI command to start a SuperNode as shown below:

		$ flower-supernode --insecure --superlink='<IP>:<PORT>'

	To view all available options, run:

		$ flower-supernode --help

	Using `start_client()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
	Instead, use the `flower-supernode` CLI command to start a SuperNode as shown below:

		$ flower-supernode --insecure --superlink='<IP>:<PORT>'

	To view all available options, run:

		$ flower-supernode --help

	Using `start_client()` is deprecated.


In [3]:
import sqlite3
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score

# 1. Ensure device and architecture are defined
device = torch.device("cpu")

class DiagnosticNet(nn.Module):
    def __init__(self):
        super(DiagnosticNet, self).__init__()
        self.layer1 = nn.Linear(30, 16)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(16, 8)
        self.output = nn.Linear(8, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.sigmoid(self.output(x))
        return x

# Instantiate a diagnostic model for evaluation
eval_model = DiagnosticNet().to(device)

print("--- Federated Diagnostic Model Node Validation ---")

total_correct = 0
total_samples = 0

for hospital_name in ["hospital_A", "hospital_B", "hospital_C"]:
    # Query isolated hospital SQLite database
    conn = sqlite3.connect(f"privacare_dbs/{hospital_name}.db")
    test_df = pd.read_sql("SELECT * FROM test_data", conn)
    conn.close()
    
    # Extract clinical features and ground-truth targets
    X_test = torch.tensor(test_df.drop(columns=["target"]).values, dtype=torch.float32).to(device)
    y_true = test_df["target"].values
    
    eval_model.eval()
    with torch.no_grad():
        preds_proba = eval_model(X_test).cpu().numpy().flatten()
        preds = (preds_proba >= 0.5).astype(int)
        
    acc = accuracy_score(y_true, preds)
    correct_count = (preds == y_true).sum()
    total_correct += correct_count
    total_samples += len(y_true)
    
    print(f"[{hospital_name}] Test Records: {len(y_true)} | Accuracy: {acc * 100:.2f}%")

global_accuracy = (total_correct / total_samples) * 100
print(f"\nConsolidated Test Accuracy: {global_accuracy:.2f}%")

--- Federated Diagnostic Model Node Validation ---
[hospital_A] Test Records: 38 | Accuracy: 31.58%
[hospital_B] Test Records: 38 | Accuracy: 44.74%
[hospital_C] Test Records: 38 | Accuracy: 42.11%

Consolidated Test Accuracy: 39.47%


In [4]:
import pandas as pd
import sqlite3
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device('cpu')


# 1. Define Architecture
class DiagnosticNet(nn.Module):

  def __init__(self):
    super().__init__()
    self.layer1 = nn.Linear(30, 16)
    self.relu = nn.ReLU()
    self.layer2 = nn.Linear(16, 8)
    self.output = nn.Linear(8, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, x):
    return self.sigmoid(self.output(self.relu(self.layer2(self.relu(self.layer1(x))))))


# 2. Load all 3 hospital databases into memory
trained_model = DiagnosticNet().to(device)
optimizer = optim.Adam(trained_model.parameters(), lr=0.01)
criterion = nn.BCELoss()

print("--- Simulating Proper Federated Training with Feature Scaling ---")

# Pull and scale data cleanly across nodes
for round_num in range(1, 4):
  for hospital_name in ['hospital_A', 'hospital_B', 'hospital_C']:
    conn = sqlite3.connect(f'privacare_dbs/{hospital_name}.db')
    train_df = pd.read_sql('SELECT * FROM train_data', conn)
    conn.close()

    X_raw = train_df.drop(columns=['target']).values
    y_raw = train_df['target'].values

    # Simple normalization so features aren't skewed
    X_norm = (X_raw - X_raw.mean(axis=0)) / (X_raw.std(axis=0) + 1e-7)

    X_train = torch.tensor(X_norm, dtype=torch.float32).to(device)
    y_train = torch.tensor(y_raw, dtype=torch.float32).view(-1, 1).to(device)

    # Local training
    trained_model.train()
    for _ in range(5):
      optimizer.zero_grad()
      out = trained_model(X_train)
      loss = criterion(out, y_train)
      loss.backward()
      optimizer.step()

  print(f'Completed Federated Round {round_num}/3')

# 3. Test the genuinely trained model
print('\n--- Real Accuracy on Test Patients ---')
total_correct = 0
total_samples = 0

for hospital_name in ['hospital_A', 'hospital_B', 'hospital_C']:
  conn = sqlite3.connect(f'privacare_dbs/{hospital_name}.db')
  test_df = pd.read_sql('SELECT * FROM test_data', conn)
  conn.close()

  X_raw = test_df.drop(columns=['target']).values
  y_true = test_df['target'].values
  X_norm = (X_raw - X_raw.mean(axis=0)) / (X_raw.std(axis=0) + 1e-7)

  X_test = torch.tensor(X_norm, dtype=torch.float32).to(device)

  trained_model.eval()
  with torch.no_grad():
    preds = (trained_model(X_test).numpy().flatten() >= 0.5).astype(int)

  correct = (preds == y_true).sum()
  acc = (correct / len(y_true)) * 100
  total_correct += correct
  total_samples += len(y_true)

  print(f'[{hospital_name}] Accuracy: {acc:.2f}%')

print(f'\nFinal Global Accuracy: {(total_correct / total_samples) * 100:.2f}%')

--- Simulating Proper Federated Training with Feature Scaling ---
Completed Federated Round 1/3
Completed Federated Round 2/3
Completed Federated Round 3/3

--- Real Accuracy on Test Patients ---
[hospital_A] Accuracy: 100.00%
[hospital_B] Accuracy: 97.37%
[hospital_C] Accuracy: 100.00%

Final Global Accuracy: 99.12%
